In [23]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input/'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/shree1992/housedata/output.csv
/kaggle/input/datasets/shree1992/housedata/data.csv
/kaggle/input/datasets/shree1992/housedata/data.dat


# Thư viện sử dụng

In [24]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import time
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, f_regression, SelectFromModel
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split

# Tiền xử lý dữ liệu

#### Tiền xử lý dữ liệu (Data Preprocessing) là quá trình làm sạch và biến đổi dữ liệu thô ban đầu thành một định dạng chuẩn xác, đồng nhất mà các thuật toán học máy có thể "đọc hiểu" và huấn luyện hiệu quả. Dữ liệu thực tế thường lộn xộn, có nhiều sai sót, do đó bước này quyết định rất lớn đến độ chính xác của kết quả dự đoán cuối cùng.

### 1. ĐỌC DỮ LIỆU

In [25]:
file_path = '/kaggle/input/datasets/shree1992/housedata/data.csv' 
try:
    df = pd.read_csv(file_path)
    print("Đọc dữ liệu thành công! Kích thước:", df.shape)
except FileNotFoundError:
    print(f"Không tìm thấy file tại {file_path}. Vui lòng kiểm tra lại đường dẫn từ ô code đầu tiên.")

Đọc dữ liệu thành công! Kích thước: (4600, 18)


### 2. XÁC ĐỊNH BIẾN MỤC TIÊU VÀ FEATURES

In [26]:
target_col = 'price' 

if target_col in df.columns:
    X = df.drop(columns=[target_col])
    y = df[target_col]
else:
    X = df.copy()
    y = None
    print(f"Lưu ý: Không tìm thấy cột mục tiêu '{target_col}'. Hãy cập nhật lại tên cột biến phụ thuộc (y).")

### 3. PHÂN TÁCH CÁC CỘT DỮ LIỆU SỐ VÀ PHÂN LOẠI

In [27]:
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X.select_dtypes(include=['object', 'category']).columns

print(f"Số lượng cột Dạng Số: {len(numeric_features)}")
print(f"Số lượng cột Dạng Phân Loại: {len(categorical_features)}")

Số lượng cột Dạng Số: 12
Số lượng cột Dạng Phân Loại: 5


### 4. TẠO PIPELINE TIỀN XỬ LÝ (PREPROCESSING PIPELINE)

In [28]:
# Pipeline cho dữ liệu số
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Pipeline cho dữ liệu phân loại
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Kết hợp cả 2 quy trình bằng ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

### 5. THỰC THI TIỀN XỬ LÝ DỮ LIỆU

In [29]:
# CHIA TẬP DỮ LIỆU TRƯỚC TIÊN
# Dùng dữ liệu X và y gốc (chưa qua biến đổi)
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# THỰC THI TIỀN XỬ LÝ
X_train = preprocessor.fit_transform(X_train_raw)

X_test = preprocessor.transform(X_test_raw)

print("Kích thước tập Train sau xử lý:", X_train.shape)
print("Kích thước tập Test sau xử lý:", X_test.shape)

Kích thước tập Train sau xử lý: (3680, 3839)
Kích thước tập Test sau xử lý: (920, 3839)


# Đánh giá hiệu suất của các kỹ thuật Feature Selection với các mô hình cơ bản

### CÁC MÔ HÌNH CƠ BẢN

#### - SVM(Support Vector Machine): Tìm ra một ranh giới (siêu phẳng) tối ưu nhất để dự đoán giá trị, chấp nhận một khoảng sai số nhất định xung quanh ranh giới đó. Nó hoạt động tốt bằng cách phạt các điểm dữ liệu nằm ngoài khoảng sai số cho phép.
#### - Random Forest: Xây dựng nhiều "cây quyết định" một cách độc lập từ các phần dữ liệu ngẫu nhiên, sau đó lấy trung bình kết quả của tất cả các cây. Cách tiếp cận này giúp mô hình giảm thiểu hiện tượng học vẹt (quá khớp) và dự đoán ổn định hơn.
#### - XGBoost(Extreme Gradient Boosting): Tạo ra chuỗi các "cây quyết định" liên tiếp nhau, trong đó cây sau sẽ tập trung sửa chữa các lỗi dự đoán của cây trước đó. Phương pháp tối ưu hóa liên tục này giúp mô hình đạt độ chính xác cực kỳ cao và tốc độ xử lý nhanh chóng.

In [30]:
models = {
    'SVM (Linear)': SVR(kernel='linear'), # Dùng kernel linear cho dữ liệu nhiều chiều
    'RandomForest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'XGBoost': XGBRegressor(n_estimators=100, random_state=42, objective='reg:squarederror', n_jobs=-1)
}

### 3. ĐỊNH NGHĨA CÁC PHƯƠNG PHÁP FEATURE SELECTION

In [31]:
# Giữ lại top 50 đặc trưng quan trọng nhất để xem mô hình cải thiện ra sao so với 4729 đặc trưng ban đầu
k_features = 50 

feature_selectors = {
    '1. All Features (Baseline)': 'passthrough', # Không chọn lọc, dùng toàn bộ ~4700 cột
    
    '2. Filter Method (SelectKBest)': SelectKBest(score_func=f_regression, k=k_features),
    
    '3. Embedded Method (RF Base)': SelectFromModel(
        RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1), 
        max_features=k_features
    )
}

### 4. VÒNG LẶP HUẤN LUYỆN VÀ ĐÁNH GIÁ

In [32]:
results = []

print("Bắt đầu tiến trình huấn luyện và đánh giá. Quá trình này có thể mất vài phút...\n")

for selector_name, selector in feature_selectors.items():
    print(f" Đang xử lý dữ liệu với: {selector_name}")
    
    # Áp dụng Feature Selection
    if selector == 'passthrough':
        X_train_sel = X_train
        X_test_sel = X_test
    else:
        # Fit trên tập train và transform cả train + test để tránh rò rỉ dữ liệu (Data Leakage)
        X_train_sel = selector.fit_transform(X_train, y_train)
        X_test_sel = selector.transform(X_test)
        
    num_features_used = X_train_sel.shape[1]

    for model_name, model in models.items():
        # Bắt đầu đo thời gian
        start_time = time.time()
        
        # Huấn luyện
        model.fit(X_train_sel, y_train)
        
        # Dự đoán
        y_pred = model.predict(X_test_sel)
        
        # Kết thúc đo thời gian
        run_time = time.time() - start_time
        
        # Tính toán các chỉ số đánh giá Regression
        r2 = r2_score(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        mae = mean_absolute_error(y_test, y_pred)
        
        # Ghi nhận kết quả
        results.append({
            'Model': model_name,
            'Feature Selection': selector_name,
            'Num Features': num_features_used,
            'R2 Score': r2,
            'RMSE': rmse,
            'MAE': mae,
            'Time (s)': round(run_time, 2)
        })

Bắt đầu tiến trình huấn luyện và đánh giá. Quá trình này có thể mất vài phút...

 Đang xử lý dữ liệu với: 1. All Features (Baseline)
 Đang xử lý dữ liệu với: 2. Filter Method (SelectKBest)
 Đang xử lý dữ liệu với: 3. Embedded Method (RF Base)


### 5. HIỂN THỊ BẢNG KẾT QUẢ TỔNG HỢP

In [33]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(by=['Model', 'R2 Score'], ascending=[True, False]).reset_index(drop=True)

print("\n" + "="*70)
print("BẢNG ĐÁNH GIÁ HIỆU SUẤT CÁC MÔ HÌNH VÀ KỸ THUẬT FEATURE SELECTION")
print("="*70)
display(results_df)


BẢNG ĐÁNH GIÁ HIỆU SUẤT CÁC MÔ HÌNH VÀ KỸ THUẬT FEATURE SELECTION


,Model,Feature Selection,Num Features,R2 Score,RMSE,MAE,Time (s)
0,RandomForest,1. All Features (Baseline),3839,0.049073,9.847840e+05,157354.232314,14.60
1,RandomForest,3. Embedded Method (RF Base),50,0.047240,9.857328e+05,164047.589215,1.72
2,RandomForest,2. Filter Method (SelectKBest),50,0.040026,9.894575e+05,180386.737019,1.14
3,SVM (Linear),1. All Features (Baseline),3839,-0.010849,1.015338e+06,264549.916041,47.71
4,SVM (Linear),3. Embedded Method (RF Base),50,-0.010860,1.015343e+06,264572.158879,0.69
5,SVM (Linear),2. Filter Method (SelectKBest),50,-0.010902,1.015364e+06,264612.720876,0.75
6,XGBoost,1. All Features (Baseline),3839,0.058243,9.800242e+05,144922.599010,1.86
7,XGBoost,3. Embedded Method (RF Base),50,0.050556,9.840158e+05,162884.846718,0.16
8,XGBoost,2. Filter Method (SelectKBest),50,0.040997,9.889570e+05,174747.444430,0.13
